# Colab setup: Search_AI_project (OTTO recsys)

Run this once at the start of every Colab session. It:
1. Mounts Drive and clones/syncs the repo there (so code + git history persist across sessions)
2. Installs the same Python environment as local (via `uv`)
3. Sets up Kaggle credentials and downloads the **full** OTTO dataset (~11GB) to Colab's local
   disk (`/content/otto_data`) - **not** Drive, since most free accounts only have 15GB of Drive
   storage and the raw dataset alone would nearly fill it. Local Colab disk is fast but wiped
   when the runtime recycles, so this cell re-downloads each session (a few minutes on Colab's
   connection) - a deliberate tradeoff, not an oversight.

Before running: make sure you have a GPU runtime (Runtime -> Change runtime type -> GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shaliqsv/Search_AI_project.git"
REPO_ROOT = Path('/content/drive/MyDrive/search_ai_project')

if not (REPO_ROOT / "pyproject.toml").exists():
    print(f"Repo not found at {REPO_ROOT} - cloning ...")
    !git clone "{REPO_URL}" "{REPO_ROOT}"
else:
    # see financebench_project's setup cell for why: self-heals a detached HEAD,
    # and --no-rebase --no-edit self-heals a diverged local/origin history
    # without hanging on an editor prompt inside a notebook `!` cell
    print(f"Repo already present at {REPO_ROOT} - syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

print(f"REPO_ROOT: {REPO_ROOT}")

## Install the environment

Colab already ships numpy/pandas/torch. We only need to add what it doesn't have. Installing
straight into Colab's own environment rather than a `uv` venv on the Drive mount deliberately -
a venv is thousands of small files, and Drive's FUSE mount is slow and occasionally flaky for
that pattern. The whole Colab runtime is already disposable/isolated, so skipping the extra
venv layer is the standard approach here, not a shortcut.

In [ ]:
!pip install -q polars duckdb lightgbm kaggle

import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
print("environment ready")

## Kaggle credentials

Uses Colab's built-in Secrets manager (key icon in the left sidebar) rather than uploading
`kaggle.json` into Drive, so your API key never sits in a synced file. Add two secrets once:
`KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com/settings -> API -> Create New Token, open the
downloaded `kaggle.json` to see both values).

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Kaggle credentials set")

## Download the full OTTO dataset

Local Colab disk, not Drive (see the note at the top). CC BY 4.0, sourced from the dataset
endpoint not the competition - see `_doc/decisions/D1-otto-license.md` for why that distinction
matters.

In [ ]:
DATA_DIR = Path('/content/otto_data')
DATA_DIR.mkdir(exist_ok=True)

!kaggle datasets download -d otto/recsys-dataset -p "{DATA_DIR}" --unzip

!ls -lh "{DATA_DIR}"

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Next steps

With the full dataset local at `/content/otto_data/otto-recsys-train.jsonl`, the Phase 1 notebook's
sampling step can either be skipped (use the full file directly, now that RAM/disk aren't the
constraint they were locally) or re-run at a much higher keep-percentage. From there, Phase 1-7
notebooks port over unchanged (same `src/ranking` code, same logic) - the only difference is scale
and that `ROOT` now resolves to this Drive-mounted repo instead of the local Mac path.

Two-Tower training (PyTorch, GPU) is the next real piece of new work once the data's in place.